# fits_history — Usage Demo
This notebook demonstrates the core functions of the fits_history library.

## Setup
Create a FITS file with a basic header to work with.

In [1]:
from astropy.io import fits
import numpy as np

from fits_history.writer import write_entry, write_entries
from fits_history.parser import parse_history
from fits_history.validator import validate_header, print_report
from fits_history.schemas import register_entry_type, unregister_entry_type, EntryType, Field, list_entry_types

# Create a simple FITS file
data = np.zeros((100, 100), dtype=np.float32)
hdu = fits.PrimaryHDU(data)
header = hdu.header

print(f"Empty header created. HISTORY cards: {len(header.get('HISTORY', []))}")

Empty header created. HISTORY cards: 0


---
## 1. write_entry — Writing Single Entries
Write one structured HISTORY entry at a time.

In [2]:
# A simple FILTER entry — fits in one card
write_entry(header, 'FILTER', {
    'TYPE': 'gaussian',
    'SIZE': 3.5,
    'AXIS': 'both'
})

# A SOFTWARE entry
write_entry(header, 'SOFTWARE', {
    'NAME': 'MyPipeline',
    'VER': '2.0.0',
    'LANG': 'Python'
})

print("Two entries written.")
print()

# Check what was written
for h in header.get('HISTORY', []):
    print(f"  {h}")

Two entries written.

  #FILTER TYPE=gaussian SIZE=3.5 AXIS=both
  #SOFTWARE NAME=MyPipeline VER=2.0.0 LANG=Python


---
## 2. write_entries — Writing Multiple Entries at Once
Write several entries in a single call. This is where continuation (`##`) happens automatically when an entry is too long for one 80-character card.

In [3]:
# This ORIGIN entry has many fields — it will overflow one card
# and automatically use ## continuation

write_entries(header, [
    ('ORIGIN', {
        'TEL': 'VLT',
        'INST': 'HAWK-I',
        'SITE': 'Paranal',
        'OBS': 'J. Smith',
        'DATE': '2024-01-15T14:30:00',
        'PROG': '098.A-0123'
    }),
    ('PARENT', {'FILE': 'raw_001.fits', 'ROLE': 'science'}),
    ('PARENT', {'FILE': 'raw_002.fits', 'ROLE': 'science'}),
    ('PARENT', {'FILE': 'std_star.fits', 'ROLE': 'calibration'}),
])

print("Four more entries written.")
print()

# Show all HISTORY cards — notice the ## continuation on ORIGIN
for h in header.get('HISTORY', []):
    print(f"  {h}")

Four more entries written.

  #FILTER TYPE=gaussian SIZE=3.5 AXIS=both
  #SOFTWARE NAME=MyPipeline VER=2.0.0 LANG=Python
  #ORIGIN TEL=VLT INST=HAWK-I SITE=Paranal OBS='J. Smith'
  ## DATE=2024-01-15T14:30:00 PROG=098.A-0123
  #PARENT FILE=raw_001.fits ROLE=science
  #PARENT FILE=raw_002.fits ROLE=science
  #PARENT FILE=std_star.fits ROLE=calibration


Notice how the `#ORIGIN` entry was too long for a single 80-character card, so the writer automatically split it across two cards using `##` as the continuation marker. The three `#PARENT` entries each fit on one card.

---
## 3. parse_history — Reading Entries Back
Parse the HISTORY cards back into structured Python objects.

In [4]:
# astropy strips the 'HISTORY ' prefix, so we add it back
history_cards = [f"HISTORY {h}" for h in header.get('HISTORY', [])]

entries = parse_history(history_cards)

print(f"Parsed {len(entries)} structured entries:\n")
for entry in entries:
    print(f"  #{entry.entry_type}: {entry.fields}")

Parsed 6 structured entries:

  #FILTER: {'TYPE': 'gaussian', 'SIZE': 3.5, 'AXIS': 'both'}
  #SOFTWARE: {'NAME': 'MyPipeline', 'VER': '2.0.0', 'LANG': 'Python'}
  #ORIGIN: {'TEL': 'VLT', 'INST': 'HAWK-I', 'SITE': 'Paranal', 'OBS': 'J. Smith', 'DATE': '2024-01-15T14:30:00', 'PROG': '098.A-0123'}
  #PARENT: {'FILE': 'raw_001.fits', 'ROLE': 'science'}
  #PARENT: {'FILE': 'raw_002.fits', 'ROLE': 'science'}
  #PARENT: {'FILE': 'std_star.fits', 'ROLE': 'calibration'}


In [5]:
# The continuation was merged — ORIGIN has all 6 fields
origin = entries[2]  # ORIGIN is the third entry
print(f"Telescope: {origin['TEL']}")
print(f"Observer:  {origin['OBS']}")
print(f"Date:      {origin['DATE']}")
print(f"Program:   {origin['PROG']}")

Telescope: VLT
Observer:  J. Smith
Date:      2024-01-15T14:30:00
Program:   098.A-0123


In [6]:
# Types are cast correctly
filt = entries[0]  # FILTER entry
print(f"SIZE value: {filt['SIZE']}")
print(f"SIZE type:  {type(filt['SIZE']).__name__}")
print()

# List all parent files
parents = [e['FILE'] for e in entries if e.entry_type == 'PARENT']
print(f"Parent files: {parents}")

SIZE value: 3.5
SIZE type:  float

Parent files: ['raw_001.fits', 'raw_002.fits', 'std_star.fits']


---
## 4. validate_header + print_report — Checking Compliance
Validate that all entries have their required fields and correct types.

In [7]:
results = validate_header(header)
print_report(results)

Validation Report
Entries checked: 6
Valid: 6  |  Invalid: 0

#FILTER: VALID
  All checks passed.

#SOFTWARE: VALID
  All checks passed.

#ORIGIN: VALID
  All checks passed.

#PARENT: VALID
  All checks passed.

#PARENT: VALID
  All checks passed.

#PARENT: VALID
  All checks passed.



In [8]:
# Now write a deliberately bad entry — missing required field
write_entry(header, 'FILTER', {'SIZE': 5.0})  # missing TYPE (required)

results = validate_header(header)
print_report(results)

Validation Report
Entries checked: 7
Valid: 6  |  Invalid: 1

#FILTER: VALID
  All checks passed.

#SOFTWARE: VALID
  All checks passed.

#ORIGIN: VALID
  All checks passed.

#PARENT: VALID
  All checks passed.

#PARENT: VALID
  All checks passed.

#PARENT: VALID
  All checks passed.

#FILTER: INVALID
  ERROR:   Missing required field 'TYPE'.



The validator caught that the last `#FILTER` entry is missing its required `TYPE` field.

---
## 5. register_entry_type — Adding Custom Types
Define and register a custom entry type not covered by the 13 built-ins.

In [9]:
print(f"Entry types before: {len(list_entry_types())}")
print()

# Define a custom type for image resampling
register_entry_type(EntryType(
    name='RESAMP',
    description='Image resampling.',
    fields=[
        Field('method', 'str', True, 'Resampling method'),
        Field('pixscale', 'float', True, 'Output pixel scale'),
        Field('kernel', 'str', False, 'Interpolation kernel'),
    ]
))

print(f"Entry types after:  {len(list_entry_types())}")
print(f"New type list: {list_entry_types()}")

Entry types before: 4

Entry types after:  5
New type list: ['ORIGIN', 'SOFTWARE', 'FILTER', 'PARENT', 'RESAMP']


In [10]:
# Write an entry using the custom type
write_entry(header, 'RESAMP', {
    'METHOD': 'bilinear',
    'PIXSCALE': 0.2,
    'KERNEL': 'lanczos3'
})

# Parse it back — now it casts types because the schema exists
history_cards = [f"HISTORY {h}" for h in header.get('HISTORY', [])]
entries = parse_history(history_cards)

resamp = [e for e in entries if e.entry_type == 'RESAMP'][0]
print(f"PIXSCALE: {resamp['PIXSCALE']} (type: {type(resamp['PIXSCALE']).__name__})")
print(f"METHOD:   {resamp['METHOD']}")

PIXSCALE: 0.2 (type: float)
METHOD:   bilinear


In [11]:
# Validate — the custom type is now recognized
from fits_history.validator import validate_entry
result = validate_entry(resamp)
print(result.report())

#RESAMP: VALID
  All checks passed.


---
## 6. unregister_entry_type — Removing Custom Types

In [12]:
print(f"Before removal: {len(list_entry_types())} types")

unregister_entry_type('RESAMP')

print(f"After removal:  {len(list_entry_types())} types")
print(f"RESAMP in list: {'RESAMP' in list_entry_types()}")

Before removal: 5 types
After removal:  4 types
RESAMP in list: False


In [13]:
# After unregistering, the validator no longer recognizes it
result = validate_entry(resamp)
print(result.report())

#RESAMP: VALID


In [15]:
# Built-in types cannot be removed
try:
    unregister_entry_type('FILTER')
except ValueError as e:
    print(f"Blocked: {e}")

Blocked: Cannot remove built-in entry type 'FILTER'.


In [16]:
# Non-existent types raise KeyError
try:
    unregister_entry_type('FOOBAR')
except KeyError as e:
    print(f"Not found: {e}")

Not found: "Entry type 'FOOBAR' not found in the registry."


---
## Save the FITS File
Write everything to disk and verify the HISTORY cards are in the file.

In [17]:
hdu.writeto('demo_output.fits', overwrite=True)

# Read it back from disk
hdul = fits.open('demo_output.fits')

print("HISTORY cards saved to disk:\n")
for h in hdul[0].header.get('HISTORY', []):
    print(f"  {h}")

hdul.close()

HISTORY cards saved to disk:

  #FILTER TYPE=gaussian SIZE=3.5 AXIS=both
  #SOFTWARE NAME=MyPipeline VER=2.0.0 LANG=Python
  #ORIGIN TEL=VLT INST=HAWK-I SITE=Paranal OBS='J. Smith'
  ## DATE=2024-01-15T14:30:00 PROG=098.A-0123
  #PARENT FILE=raw_001.fits ROLE=science
  #PARENT FILE=raw_002.fits ROLE=science
  #PARENT FILE=std_star.fits ROLE=calibration
  #FILTER SIZE=5.0
  #RESAMP METHOD=bilinear PIXSCALE=0.2 KERNEL=lanczos3
